# 02. Rating every advertisement with the language-model rater

**Design decisions.**
- Rater: one API model, fixed in SETTINGS (`claude-sonnet-5-5`), no up-front thinking (`thinking: between_tools`, the lowest setting that model accepts), `max_tokens` 8, one-word answer parsed on its first word (`src/rater.py`). The prompt is the six-part rubric of Table I, written out once and versioned (`rubric-v1`). The current API models reject sampling parameters, so there is no temperature setting; stability is measured instead: every advertisement is rated twice and a third pass breaks ties.
- Every call is appended to `data/ratings/cache.jsonl` keyed by (model, prompt version, advertisement id, pass) with the served model string, token counts and the raw answer. Re-running any cell never repeats a paid call.
- Spend is projected before anything runs (Stage 2). Stage 3 rates three advertisements once and shows the raw answers. Stage 4 runs only after `RUN_FULL = True` is set in SETTINGS. `DRY_RUN = True` replaces the API with a keyword stand-in to check the plumbing at zero cost; dry-run rows are cached under a different model name and never mix with real ones.
- Advertisements rated: the 2019 pool and the rebuilt 2024 and 2026 pools from notebook 01 (if a pool does not exist yet, the June subset stands in with a printed warning). Within each (year, route) stratum larger than CAP_PER_STRATUM a fixed random sample of that size is rated and the rest is left unrated; the full stratum sizes go to `reports/pool_sizes.csv` and the draw to `reports/rating_sample.json`, so notebook 04 can reweight. The June 2024 sample (120) and June 2026 subset (56) are always rated as checks: the first against the June labels (`reports/rater_comparison_june_vs_api_2024.json`), the second against the rebuilt 2026 pool.
- Outputs: `reports/ratings_<model>.csv` (id, year, source, title, route, passes, final, flags, served model, tokens; no text), `reports/rating_summary.json` (pass agreement, ties, malformed answers, token totals and spend at list prices, share by year and route).


In [ ]:
# === SETTINGS ===
import os
from pathlib import Path
REPO_URL   = "https://github.com/anasbiswas1/hiring-research.git"
DRIVE_ROOT = Path(os.environ.get("HIRING_DRIVE_ROOT", "/content/drive/MyDrive/Hiring_Research"))
REPO_DIR   = DRIVE_ROOT / "hiring-research"
MODEL           = "claude-sonnet-5-5"
PASSES          = 2
OPENING_CHARS   = 700
SMOKE_N         = 3
CAP_PER_STRATUM = 500     # fixed random sample per (year, route) when a pool is larger; the rest is not rated
CAP_SEED        = 7
DRY_RUN       = False     # True: keyword stand-in instead of the API, zero cost, plumbing check only
RUN_FULL      = False     # set True after reading the smoke test output; nothing is billed before that


In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, shutil, subprocess, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CRED = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["HIRING_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["HIRING_REPO_ROOT"] = str(REPO_DIR)
assert (REPO_DIR / ".git").exists(), f"no git checkout at {REPO_DIR}: run notebook 00 Stage 0 first"
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)


## Stage 0. Install

In [ ]:
# === INSTALL ===
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['anthropic', 'pandas']], capture_output=True, text=True)
print("pip -> rc", r.returncode)
print(r.stderr[-600:])


## Stage 1. API client
The key is asked for with getpass and kept in the session environment only.

In [ ]:
# === STAGE 1: API client ===
import os, getpass, importlib
import rater
importlib.reload(rater)
if DRY_RUN:
    client = rater.FakeClient()
    RUN_MODEL = "dry-run-" + MODEL
    print("DRY RUN: keyword stand-in, no API calls, cached under", RUN_MODEL)
else:
    import anthropic
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (kept in this session only): ")
    client = anthropic.Anthropic()
    RUN_MODEL = MODEL
    print("API client ready; model", RUN_MODEL)


## Stage 2. Advertisements to rate and the spend projection

In [ ]:
# === STAGE 2: advertisements to rate (pools capped per stratum, June subsets as checks), spend projection ===
import importlib
import numpy as np
import pandas as pd
import ads, rater, results
importlib.reload(ads); importlib.reload(rater); importlib.reload(results)
SUB = REPO_DIR / "data" / "subsets"

def load(name):
    return pd.read_csv(SUB / f"{name}.csv", dtype={"ad_id": str})

pools, checks = [load("junior_2019")], []
for year, pool_name, check_name in [(2024, "junior_2024", "june_2024_sample"), (2026, "junior_2026", "june_2026_subset")]:
    chk = load(check_name)
    chk = chk.drop(columns=[c for c in ("june_senior",) if c in chk.columns])
    if (SUB / f"{pool_name}.csv").exists():
        pools.append(load(pool_name)); checks.append(chk)
        print(f"{year} pool from notebook 01: {len(pools[-1])}")
    else:
        pools.append(chk)
        print(f"WARNING: {pool_name}.csv not found (notebook 01 not run); {check_name} stands in as the {year} pool")
POOL = pd.concat(pools, ignore_index=True).drop_duplicates("ad_id")
POOL["route"] = POOL.why.map(results.route_of)
sizes = POOL.groupby(["year", "route"]).size().rename("N").reset_index()
sizes.to_csv(REPO_DIR / "reports" / "pool_sizes.csv", index=False)

rng = np.random.default_rng(CAP_SEED)
POOL["in_pool"] = True
sample_log = []
for (y, r), g in POOL.groupby(["year", "route"]):
    n = len(g)
    if n > CAP_PER_STRATUM:
        drop = g.index.to_numpy()[rng.permutation(n)[CAP_PER_STRATUM:]]
        POOL.loc[drop, "in_pool"] = False
    sample_log.append({"year": int(y), "route": r, "N": int(n), "rated": int(min(n, CAP_PER_STRATUM))})
ads.write_json(REPO_DIR / "reports" / "rating_sample.json", {"cap_per_stratum": CAP_PER_STRATUM, "seed": CAP_SEED, "strata": sample_log})
print(pd.DataFrame(sample_log).to_string(index=False))

CHECK = pd.concat(checks, ignore_index=True) if checks else pd.DataFrame(columns=POOL.columns)
CHECK = CHECK[~CHECK.ad_id.isin(set(POOL[POOL.in_pool].ad_id))].copy()   # June advertisements not rated as pool members, including any sampled out
CHECK["route"] = CHECK.why.map(results.route_of); CHECK["in_pool"] = False
ADS = pd.concat([POOL[POOL.in_pool], CHECK], ignore_index=True)
print(f"to rate: {len(ADS)} advertisements ({int(POOL.in_pool.sum())} in the pools, {len(CHECK)} June check advertisements not in the pools)")
est = rater.estimate_cost(ADS, PASSES, MODEL, OPENING_CHARS)
tie = rater.estimate_cost(ADS, 1, MODEL, OPENING_CHARS)
print("projected:", est["calls"], "calls,", est["input_tokens"], "input tokens, about $%.2f" % est["usd_projected"],
      "(+ up to $%.2f if every advertisement needed a tie-break; expect a few percent of that)" % tie["usd_projected"])
CACHE = rater.RatingCache(REPO_DIR / "data" / "ratings" / "cache.jsonl")
print("cache rows already present for this model:", len(CACHE.frame(RUN_MODEL, rater.PROMPT_VERSION)))


## Stage 3. Smoke test (three advertisements, one pass)

In [ ]:
# === STAGE 3: smoke test, SMOKE_N advertisements, one pass, raw answers shown ===
n = rater.rate_pass(client, ADS, 1, CACHE, model=RUN_MODEL, n_chars=OPENING_CHARS, limit=SMOKE_N, progress_every=0)
f = CACHE.frame(RUN_MODEL, rater.PROMPT_VERSION)
show = f.merge(ADS[["ad_id", "title"]], on="ad_id").head(SMOKE_N)
print(f"{n} new calls; served model: {sorted(set(f.served_model))}")
print(show[["title", "verdict_raw", "verdict", "input_tokens", "output_tokens", "stop_reason"]].to_string(index=False))
assert show.verdict.notna().all(), "an answer was not YES or NO: read verdict_raw before spending more"
print("\nIf the answers look like one-word verdicts, set RUN_FULL = True in SETTINGS and run Stage 4.")


## Stage 4. Full run (gated by RUN_FULL)

In [ ]:
# === STAGE 4: full run, two passes, tie-break ===
assert RUN_FULL, "set RUN_FULL = True in SETTINGS after checking the smoke test; nothing has been billed beyond the smoke test"
calls = 0
for p in range(1, PASSES + 1):
    calls += rater.rate_pass(client, ADS, p, CACHE, model=RUN_MODEL, n_chars=OPENING_CHARS)
    print(f"pass {p} complete; cumulative new calls this session: {calls}")
FIN = rater.finalize(CACHE, ADS, model=RUN_MODEL)
ties = rater.needs_tie_break(FIN)
print(f"advertisements needing a third pass: {len(ties)}")
if ties:
    calls += rater.rate_pass(client, ADS, 3, CACHE, model=RUN_MODEL, n_chars=OPENING_CHARS, only_ad_ids=ties, progress_every=0)
    FIN = rater.finalize(CACHE, ADS, model=RUN_MODEL)
still = FIN[FIN.final.isna()]
if len(still):
    print("WARNING: still undecided after three passes (treated as missing; listed):")
    print(still[["ad_id", "title", "pass1", "pass2", "pass3"]].to_string(index=False))
print("new calls this session:", calls)
print(rater.stability(FIN))


## Stage 5. Reports and the comparison with the June labels

In [ ]:
# === STAGE 5: reports (no advertisement text), the June comparisons ===
import json
import ads, validation
import pandas as pd
FIN = rater.finalize(CACHE, ADS, model=RUN_MODEL)
FIN = FIN.merge(ADS[["ad_id", "in_pool", "why", "route"]], on="ad_id", how="left")
out = REPO_DIR / "reports" / f"ratings_{RUN_MODEL}.csv"
FIN.to_csv(out, index=False)
f = CACHE.frame(RUN_MODEL, rater.PROMPT_VERSION)
p_in, p_out = rater.PRICES.get(MODEL, (0, 0))
pool_fin = FIN[FIN.in_pool].dropna(subset=["final"])
summary = {"model": RUN_MODEL, "prompt_version": rater.PROMPT_VERSION, "served_models": sorted(set(f.served_model)),
           "calls": int(len(f)), "input_tokens": int(f.input_tokens.sum()), "output_tokens": int(f.output_tokens.sum()),
           "usd_at_list_price": round(f.input_tokens.sum() / 1e6 * p_in + f.output_tokens.sum() / 1e6 * p_out, 3),
           "stability": rater.stability(FIN),
           "share_by_year_route_in_pool": {f"{y}|{r}": {"n": int(len(g)), "share": round(float(g.final.astype(float).mean()), 4)}
                                           for (y, r), g in pool_fin.groupby(["year", "route"])}}
ads.write_json(REPO_DIR / "reports" / "rating_summary.json", summary)
print(json.dumps(summary, indent=2))

june24 = pd.read_csv(REPO_DIR / "data" / "subsets" / "june_2024_sample.csv", dtype={"ad_id": str})
cmp = june24[["ad_id", "title", "june_senior", "why"]].merge(FIN[["ad_id", "final"]], on="ad_id", how="inner").dropna(subset=["final"])
res = {"n": int(len(cmp)), "june_share": round(float(cmp.june_senior.mean()), 4), "api_share": round(float(cmp.final.astype(int).mean()), 4),
       "agreement_june_vs_api": validation.agreement(cmp.june_senior, cmp.final.astype(int)),
       "by_route": {k: {"n": int(len(g)), "june_share": round(float(g.june_senior.mean()), 3), "api_share": round(float(g.final.astype(int).mean()), 3)}
                    for k, g in cmp.groupby("why")}}
ads.write_json(REPO_DIR / "reports" / "rater_comparison_june_vs_api_2024.json", res)
cmp.rename(columns={"final": "api_final"}).to_csv(REPO_DIR / "reports" / "rater_comparison_june_vs_api_2024.csv", index=False)
print("June rater vs API rater on the June 2024 sample:", json.dumps(res, indent=2))

june26 = pd.read_csv(REPO_DIR / "data" / "subsets" / "june_2026_subset.csv", dtype={"ad_id": str})
c26 = june26[["ad_id", "why"]].merge(FIN[["ad_id", "final"]], on="ad_id", how="inner").dropna(subset=["final"])
chk26 = {"n": int(len(c26)), "api_share": round(float(c26.final.astype(int).mean()), 4) if len(c26) else None,
         "by_route": {k: {"n": int(len(g)), "api_share": round(float(g.final.astype(int).mean()), 3)} for k, g in c26.groupby("why")}}
ads.write_json(REPO_DIR / "reports" / "june_2026_subset_check.json", chk26)
print("June 2026 Kaggle subset rated by the same rater (consistency check):", json.dumps(chk26))


## End of unit: commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "symbolic-ref", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb02: pools rated twice with the fixed rubric prompt (capped per year and route), ties broken; reports/ratings_<model>.csv, rating_summary.json, rating_sample.json, pool_sizes.csv, rater_comparison_june_vs_api_2024.json, june_2026_subset_check.json'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
